In [2]:
import pandas as pd
import numpy as np

# Load Data

In [4]:
data_d = '/projects/MARGULIS/youtube-music-comments/data/YouNiverse/'
df_sb_f = pd.read_csv(f"{data_d}/_raw_df_timeseries.tsv.gz", compression="infer", sep="\t")
df_ch_f = pd.read_csv(f"{data_d}/_raw_df_channels.tsv.gz", compression="infer", sep="\t")
df_vd_f = pd.read_feather(f"{data_d}/yt_metadata_helper.feather")
# df_sb_f = pd.read_csv(f"{data_d}/_raw_df_timeseries.tsv.gz", compression="infer", sep="\t")
df_ch_f_en = pd.read_csv(f"{data_d}/df_channels_en.tsv.gz", compression="infer", sep="\t")

df_vd_f["dummmy"] = 1
df_sb_f["datetime"] = pd.to_datetime(df_sb_f["datetime"])
df_ch_f["join_date"] = pd.to_datetime(df_ch_f["join_date"])
num_comments = pd.read_csv(f"{data_d}/num_comments.tsv.gz", compression="infer", sep="\t")
num_comments_authors = pd.read_csv(f"{data_d}/num_comments_authors.tsv.gz", compression="infer", 
                                   sep="\t")

In [ ]:
import zstandard as zstd
import io
import json

# dict_keys(['categories', 'channel_id', 'crawl_date', 'description', 'dislike_count', 'display_id', 'duration', 'like_count', 'tags', 'title', 'upload_date', 'view_count'])

input_path = '/projects/MARGULIS/youtube-music-comments/data/YouNiverse/_raw_yt_metadata.jsonl.zst'
output_path = '/projects/MARGULIS/youtube-music-comments/data/YouNiverse/music_filtered_yt_metadata.jsonl.zst'

with open(input_path, 'rb') as input_fh, open(output_path, 'wb') as output_fh:
    dctx = zstd.ZstdDecompressor()
    cctx = zstd.ZstdCompressor()
    
    with dctx.stream_reader(input_fh) as reader, cctx.stream_writer(output_fh) as writer:
        text_stream = io.TextIOWrapper(reader, encoding='utf-8')
        output_stream = io.TextIOWrapper(writer, encoding='utf-8', write_through=True)
        
        counter = 0
        for line in text_stream:
            counter += 1
            record = json.loads(line)
            if record.get('categories') == 'Music':  # Filter by 'categories'
                output_stream.write(json.dumps(record) + '\n')
            
            if counter % 10000 == 0:  # Print progress every 10,000 lines
                print(f"Processed {counter} lines")

In [4]:
df = pd.read_json('/projects/MARGULIS/youtube-music-comments/data/YouNiverse/music_filtered_yt_metadata.jsonl.zst', lines=True)

In [7]:
# Save the full DataFrame as CSV
output_path = '/projects/MARGULIS/youtube-music-comments/data/YouNiverse/music_filtered_yt_metadata.csv'
df.to_csv(output_path, index=False)

# Drop the 'description' column and save as another CSV
output_path_no_desc = '/projects/MARGULIS/youtube-music-comments/data/YouNiverse/music_filtered_yt_metadata_no_desc.csv'
df.drop(columns=['description'], inplace=False).to_csv(output_path_no_desc, index=False)


KeyboardInterrupt: 

In [ ]:
# notes: display_id is video id. (in df_vd_f and num_comments)
# categories is always there, can filter inplace
# vid count per channel: df_ch_f['videos_cc']

# filtering - only keep music

print(f'shapes before filtering: {df_sb_f.shape}, {df_ch_f.shape}, {df_vd_f.shape}')
df_sb_f = df_sb_f[df_sb_f['category'] == 'Music']
df_ch_f = df_ch_f[df_ch_f['category_cc'] == 'Music']
df_vd_f = df_vd_f[df_vd_f['categories'] == 'Music']
print(f'shapes after filtering: {df_sb_f.shape}, {df_ch_f.shape}, {df_vd_f.shape}')

shapes before filtering: (21553648, 10), (156977, 7), (72924794, 9)
shapes after filtering: (4211778, 10), (29166, 7), (8305003, 9)


## All columns in all dfs:
- Timeseries (df_sb_f): ['channel', 'category', 'datetime', 'views', 'delta_views', 'subs', 'delta_subs', 'videos', 'delta_videos', 'activity']
- Channels (df_ch_f): ['category_cc', 'join_date', 'channel', 'name_cc', 'subscribers_cc', 'videos_cc', 'subscriber_rank_sb', 'videos_yt']
- Videos (df_vd_f): ['categories', 'channel_id', 'dislike_count', 'display_id', 'duration','like_count', 'upload_date', 'view_count', 'dummmy']
- Comments (num_comments): ['display_id', 'num_comms']
- Comment authors (num_comments_authors): ['author', 'video_id']

In [14]:
# filter comments: num_comments drop rows where display_id is not in df_vd_f
print(f'shape num_comments before filtering: {num_comments.shape}')
num_comments = num_comments[num_comments['display_id'].isin(df_vd_f['display_id'])]
print(f'shape num_comments after filtering: {num_comments.shape}')

shape num_comments before filtering: (72924794, 2)
shape num_comments after filtering: (8305003, 2)


In [15]:
# Save music only data
df_sb_f.to_csv(f"{data_d}/df_timeseries_music.tsv.gz", compression="infer", sep="\t", index=False)
df_ch_f.to_csv(f"{data_d}/df_channels_music.tsv.gz", compression="infer", sep="\t", index=False)
df_vd_f.to_feather(f"{data_d}/yt_metadata_helper_music.feather")
num_comments.to_csv(f"{data_d}/num_comments_music.tsv.gz", compression="infer", sep="\t", index=False)
